# Enhanced Ecommerce EDA - Azure Synapse
Includes sales, customers, products and review analytics.

In [2]:
df = spark.read.parquet(
    "abfss://curated@stecommercedlspl.dfs.core.windows.net/"
)

display(df.limit(10))

StatementMeta(ApSparkpool1, 1, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0e0bdeb1-6e55-4da0-a1a4-4c0ce6e442a5)

In [3]:
print('Rows:', df.count())
print('Columns:', len(df.columns))
df.printSchema()

StatementMeta(ApSparkpool1, 1, 4, Finished, Available, Finished, False)

Rows: 112287
Columns: 33
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,0) (nullable = true)
 |-- freight_value: decimal(10,0) (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_score: intege

In [4]:
from pyspark.sql.functions import col,count,when
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

StatementMeta(ApSparkpool1, 1, 5, Finished, Available, Finished, False)

+-----------+------------------+------------------------+-------------+--------------+--------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+----------+---------+-------------------+-----+-------------+---------+------------+--------------------+----------------------+--------------------+-----------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|order_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|review_id|review_score|review_comment_title|review_comment_message|review_creation_date

## Top Customers by Spend

In [5]:
from pyspark.sql.functions import sum
df.groupBy('customer_id').agg(sum('total_price').alias('Spent')).orderBy('Spent', ascending=False).show(10)

StatementMeta(ApSparkpool1, 1, 6, Finished, Available, Finished, False)

+--------------------+-----+
|         customer_id|Spent|
+--------------------+-----+
|1617b1357756262bf...|13664|
|ec5b2ba62e5743423...| 7276|
|c6e2731c5b391845f...| 6929|
|3fd6777bbce08a352...| 6727|
|05455dfa7cd02f13d...| 6078|
|df55c14d1476a9a34...| 4950|
|e0a2412720e9ea4f2...| 4810|
|24bbf5fd2f2e1b359...| 4764|
|3d979689f636322c6...| 4682|
|1afc82cd60e303ef0...| 4513|
+--------------------+-----+
only showing top 10 rows



## Revenue by Product Category

In [6]:
from pyspark.sql.functions import sum
df.groupBy('product_category_name').agg(sum('total_price').alias('Revenue')).orderBy('Revenue', ascending=False).show(10)

StatementMeta(ApSparkpool1, 1, 7, Finished, Available, Finished, False)

+---------------------+-------+
|product_category_name|Revenue|
+---------------------+-------+
|         beleza_saude|1433479|
|   relogios_presentes|1297310|
|      cama_mesa_banho|1245188|
|        esporte_lazer|1154319|
| informatica_acess...|1061859|
|     moveis_decoracao| 902447|
| utilidades_domest...| 775378|
|           cool_stuff| 713067|
|           automotivo| 678374|
|   ferramentas_jardim| 581598|
+---------------------+-------+
only showing top 10 rows



## Most Ordered Products

In [7]:
from pyspark.sql.functions import count
df.groupBy('product_id').agg(count('*').alias('Orders')).orderBy('Orders', ascending=False).show(10)

StatementMeta(ApSparkpool1, 1, 8, Finished, Available, Finished, False)

+--------------------+------+
|          product_id|Orders|
+--------------------+------+
|aca2eb7d00ea1a7b8...|   524|
|422879e10f4668299...|   486|
|99a4788cb24856965...|   482|
|389d119b48cf3043d...|   390|
|368c6c730842d7801...|   388|
|53759a2ecddad2bb8...|   373|
|d1c427060a0f73f6b...|   340|
|53b36df67ebb7c415...|   320|
|154e7e31ebfa09220...|   292|
|3dd2a17168ec895c7...|   272|
+--------------------+------+
only showing top 10 rows



## Review Score Distribution 

In [8]:
if 'review_score' in df.columns:
    df.groupBy('review_score').count().orderBy('review_score').show()
else:
    print('review_score column not found')

StatementMeta(ApSparkpool1, 1, 9, Finished, Available, Finished, False)

+------------+-----+
|review_score|count|
+------------+-----+
|           1|14184|
|           2| 3868|
|           3| 9417|
|           4|21307|
|           5|63511|
+------------+-----+



## Average Rating by Product Category

In [9]:
from pyspark.sql.functions import avg
if 'review_score' in df.columns:
    df.groupBy('product_category_name').agg(avg('review_score').alias('AvgRating')).orderBy('AvgRating', ascending=False).show(10)
else:
    print('review_score column not found')

StatementMeta(ApSparkpool1, 1, 10, Finished, Available, Finished, False)

+---------------------+------------------+
|product_category_name|         AvgRating|
+---------------------+------------------+
|    cds_dvds_musicais| 4.642857142857143|
| fashion_roupa_inf...|               4.5|
| livros_interesse_...|4.4489051094890515|
| construcao_ferram...| 4.444444444444445|
|    livros_importados| 4.440677966101695|
|               flores| 4.419354838709677|
|      livros_tecnicos| 4.368421052631579|
|    alimentos_bebidas| 4.320143884892087|
|     malas_acessorios| 4.315257352941177|
| portateis_casa_fo...| 4.302631578947368|
+---------------------+------------------+
only showing top 10 rows



## Top Rated Products

In [10]:
from pyspark.sql.functions import avg,count
if 'review_score' in df.columns:
    df.groupBy('product_id').agg(avg('review_score').alias('Rating'),count('*').alias('Reviews')).orderBy('Rating', ascending=False).show(10)
else:
    print('review_score column not found')

StatementMeta(ApSparkpool1, 1, 11, Finished, Available, Finished, False)

+--------------------+------+-------+
|          product_id|Rating|Reviews|
+--------------------+------+-------+
|6a1281ddeb8279b77...|   5.0|      2|
|53e8156b28f9080cc...|   5.0|      1|
|e2a5fbcec3cf55f09...|   5.0|      1|
|05f0fe07929d35be0...|   5.0|      5|
|eb048e296b8db6cd6...|   5.0|      2|
|f42b6d7b1de09b021...|   5.0|      1|
|66e9fb73e349cba44...|   5.0|      8|
|3154b455f8ac2cef3...|   5.0|      1|
|290ada89b05e1dca2...|   5.0|      2|
|9823e38ee143a67b9...|   5.0|      1|
+--------------------+------+-------+
only showing top 10 rows



## KPI Summary

In [11]:
from pyspark.sql.functions import sum,avg,countDistinct
df.agg(sum('total_price').alias('TotalRevenue'), countDistinct('customer_id').alias('Customers'), countDistinct('order_id').alias('Orders')).show()

StatementMeta(ApSparkpool1, 1, 12, Finished, Available, Finished, False)

+------------+---------+------+
|TotalRevenue|Customers|Orders|
+------------+---------+------+
|    15763933|    97851| 97851|
+------------+---------+------+

